In [2]:
!uv pip install chromadb

Using Python 3.10.20 environment at: /Users/kbshal/mero_space/sarathi_academy/.venv
Resolved 86 packages in 1.82s                                        
Prepared 37 packages in 5.18s                                            
Installed 40 packages in 43ms                               
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.3
 + aiosignal==1.4.0
 + async-timeout==5.0.1
 + bcrypt==5.0.0
 + build==1.5.0
 + chromadb==1.5.9
 + coloredlogs==15.0.1
 + durationpy==0.10
 + flatbuffers==25.12.19
 + frozenlist==1.8.0
 + googleapis-common-protos==1.75.1
 + grpcio==1.83.0
 + httptools==0.8.0
 + humanfriendly==10.0
 + importlib-resources==7.1.0
 + kubernetes==36.0.3
 + mmh3==5.2.1
 + multidict==6.7.1
 + oauthlib==3.3.1
 + onnxruntime==1.23.2
 + opentelemetry-api==1.44.0
 + opentelemetry-exporter-otlp-proto-common==1.44.0
 + opentelemetry-exporter-otlp-proto-grpc==1.44.0
 + opentelemetry-proto==1.44.0
 + opentelemetry-sdk==1.44.0
 + opentelemetry-semantic-conventions==0.65b0
 + orjson==3.12.0

In [20]:
import numpy as np
import numpy as np
from review_data import make_reviews              # the 60 reviews from last class
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import make_pipeline

In [35]:
chunks = [
    "the food was good and the service was fast",
    "good food and very friendly staff",
    "great service and great value for money",
    "the momo was delicious and the staff were friendly",
    "delicious food, fast service, good price",
    "excellent food and excellent service",
    "the curry was delicious and the staff was kind",
    "friendly staff and fresh food",
    "fresh ingredients and good flavour",
    "the dal bhat was delicious and hot",
    "good value and the service was quick",
    "quick service and delicious coffee",
    "the staff was friendly and the food was fresh",
    "great flavour and a clean place",
    "clean tables and very good food",
    "the thukpa was excellent and hot",
    "excellent value, the food was fresh",
    "we loved the food, service was fast",
    "the biryani was delicious and the portion was generous",
    "generous portion and good price",
    "the tea was good and the staff smiled",
    "friendly waiter and delicious pastries",
    "the food arrived hot and fresh",
    "hot food, fast service, friendly staff",
    "very good experience, we will come again",
    "the chef was great and the food was delicious",
    "great place, clean and friendly",
    "the salad was fresh and the service was good",
    "good coffee and a quick, friendly staff",
    "delicious flavour and excellent price",
]


In [1]:
import chromadb

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import make_pipeline


# Create and FIT the model once
lsa = make_pipeline(
    TfidfVectorizer(stop_words="english"),
    TruncatedSVD(n_components=2, random_state=0),

)

chunk_embeddings = lsa.fit_transform(chunks)

print("chunk embeddings:", chunk_embeddings.shape)


# Query function
def embed(query):
    embedding = lsa.transform([query])
    print("query embedding:", embedding.shape)
    return embedding


# Chroma
client = chromadb.Client()

col = client.create_collection(
    "notes_20",  # use a new collection while debugging
    metadata={"hnsw:space": "cosine"}
)

col.add(
    ids=[str(i) for i in range(len(chunks))],
    embeddings=chunk_embeddings.tolist(),
    documents=chunks
)


# Search
q = "staff and fresh food"

query_embedding = embed(q)

results = col.query(
    query_embeddings=query_embedding.tolist(),
    n_results=2
)

print(results["documents"])

NameError: name 'chunks' is not defined

In [ ]:
results

In [ ]:


client = chromadb.Client()

col = client.get_collection("notes_20")


q = "staff and fresh food"

results = col.query(
    query_embeddings=query_embedding.tolist(),
    n_results=2
)


results


In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ==========================================
# STEP 1: THE DATABASE (CHUNKING)
# ==========================================
# Imagine this is a textbook that we chopped into 5 paragraphs (chunks).
# The computer will automatically assign them IDs: 0, 1, 2, 3, 4.
chunks = [
    "Overfitting happens when a machine learning model memorizes the training data.", # ID 0
    "A database is used to store data safely.",                                       # ID 1
    "The learning rate controls how big of a step the model takes during training.",  # ID 2
    "Python is a popular programming language.",                                      # ID 3
    "A transformer is a deep learning architecture using attention mechanisms."       # ID 4
]

# ==========================================
# STEP 2: THE EMBEDDER (TRANSLATION)
# ==========================================
# We create a machine to turn English words into math (numbers).
embedder = TfidfVectorizer()

# We pass all our chunks through the machine to create our "Database Embeddings".
database_embeddings = embedder.fit_transform(chunks)


# ==========================================
# STEP 3: THE SEARCH ENGINE (THE KITCHEN)
# ==========================================
def ranked(q):
    """Takes a question, does similarity math, and returns a sorted list of Chunk IDs."""
    
    # 1. Turn the user's question into math
    q_vec = embedder.transform([q])
    
    # 2. Compare the question to every chunk in the database (Cosine Similarity)
    scores = cosine_similarity(q_vec, database_embeddings)[0]
    
    # 3. Sort the scores from highest match to lowest match
    sorted_ids = np.argsort(-scores) 
    
    # 4. Hand back the list of IDs (e.g., [2, 0, 4, 1, 3])
    return sorted_ids.tolist()


# ==========================================
# STEP 4: THE ANSWER KEY (THE HUMAN GRADER)
# ==========================================
# A human read the chunks above and wrote down the perfect answers.
EVAL = [
    ("what is overfitting?", 0),                     # The answer is in Chunk 0
    ("what does the learning rate control?", 2),     # The answer is in Chunk 2
    ("what is a transformer?", 4)                    # The answer is in Chunk 4
]


# ==========================================
# STEP 5: THE METRICS (RECALL & MRR)
# ==========================================
def rank_of_gold(q, gold):
    """Finds what position the Search Engine put the correct answer in."""
    retrieved_list = ranked(q)
    
    # If the search engine totally failed, give it an infinite rank
    if gold not in retrieved_list:
        return float('inf')
        
    # Find the index of the gold chunk, and add 1 (because humans count from 1)
    return retrieved_list.index(gold) + 1


# --- Let's run the actual test! ---

k = 3 # We only have patience to look at the Top 3 results

# Notice the square brackets [] to create a completed list before doing the math!
recall_list = [rank_of_gold(q, gold) <= k for q, gold in EVAL]
mrr_list    = [1 / rank_of_gold(q, gold) for q, gold in EVAL]

final_recall = np.mean(recall_list)
final_mrr    = np.mean(mrr_list)

print("--- AI SEARCH ENGINE REPORT CARD ---")
print(f"Final Recall@{k} Score: {final_recall * 100}%")
print(f"Final MRR Score:      {final_mrr}")

In [ ]:
q = "what is overfitting?"

def ranked(q):
    """Takes a question, does similarity math, and returns a sorted list of Chunk IDs."""
    
    # 1. Turn the user's question into math
    q_vec = embedder.transform([q])
    
    # 2. Compare the question to every chunk in the database (Cosine Similarity)
    scores = cosine_similarity(q_vec, database_embeddings)[0]
    
    # 3. Sort the scores from highest match to lowest match
    sorted_ids = np.argsort(-scores) 
    
    # 4. Hand back the list of IDs (e.g., [2, 0, 4, 1, 3])
    return sorted_ids.tolist()


    

# NEXT_LOGIC

In [17]:
import chromadb
from sklearn.feature_extraction.text import TfidfVectorizer

# ==========================================
# STEP 1: THE DATA & METADATA (STICKY NOTES)
# ==========================================
chunks = [
    "Oak trees grow very tall and drop leaves.",          # ID 0
    "Pine trees keep their green needles all year.",      # ID 1
    "To get stronger, you must lift heavy weights.",      # ID 2
    "Cardio training improves your heart health.",        # ID 3
    "A transformer is an advanced AI architecture."       # ID 4
]

# We attach a "sticky note" (metadata) to every single chunk
metadatas = [
    {"topic": "trees"},       # Matches Chunk 0
    {"topic": "trees"},       # Matches Chunk 1
    {"topic": "training"},    # Matches Chunk 2
    {"topic": "training"},    # Matches Chunk 3
    {"topic": "ai"}           # Matches Chunk 4
]

# ==========================================
# STEP 2: THE EMBEDDER (TRANSLATOR)
# ==========================================
vectorizer = TfidfVectorizer()
# Convert chunks to math, and turn it into a standard dense array
chunk_embeddings = vectorizer.fit_transform(chunks).toarray() 

def embed(text):
    """Helper function to translate a single question into math"""
    return vectorizer.transform([text]).toarray()[0]


# ==========================================
# STEP 3: SETTING UP THE DATABASE
# ==========================================
client = chromadb.Client() # Opens an in-memory database

# Create a collection (like a table in a database)
col = client.create_collection(
    name="notes_2",
    metadata={"hnsw:space": "cosine"}  # Tell it to use Cosine Similarity!
)


# ==========================================
# STEP 4: INGESTION (FILLING THE CABINET)
# ==========================================
# Create ID strings: ['0', '1', '2', '3', '4']
ids = [str(i) for i in range(len(chunks))]

# Hand everything over to Chroma. It organizes it instantly.
col.add(
    ids=ids,
    embeddings=chunk_embeddings.tolist(),
    documents=chunks,
    metadatas=metadatas
)


# ==========================================
# STEP 5: THE SEARCH (WITH A FILTER!)
# ==========================================
question = "How do I build muscle?"

# Search the database, but strictly limit it to the "training" topic
results = col.query(
    query_embeddings=[embed(question).tolist()],
    n_results=2,
    where={"topic": "training"}  # The Magic Filter!
)

print("--- SEARCH RESULTS ---")
# Chroma returns a dictionary of lists, so we grab the first list of documents [0]
for doc in results['documents'][0]:
    print(f"- {doc}")

--- SEARCH RESULTS ---
- To get stronger, you must lift heavy weights.
- Cardio training improves your heart health.


In [18]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ==========================================
# STEP 1: THE DATABASE
# ==========================================
documents = [
    "The quick sports car",
    "A fast food delivery vehicle",
    "The fastest racing vehicle"
]

# ==========================================
# STEP 2: THE TWO DETECTIVES
# ==========================================
# Detective Sparse: Only cares about exact letter-for-letter keyword matches.
sparse_machine = CountVectorizer()
sparse_db = sparse_machine.fit_transform(documents)

# Detective Dense: Cares about the "weight" and meaning of the words (TF-IDF).
# (In a real system, you would use SVD or an LLM Embedding here).
dense_machine = TfidfVectorizer()
dense_db = dense_machine.fit_transform(documents)


# ==========================================
# STEP 3: THE NORMALIZER (The Peacemaker)
# ==========================================
def norm(scores):
    """
    Forces all scores to live perfectly between 0.0 and 1.0.
    If the highest score is 50, it divides everything by 50 so the winner is 1.0.
    """
    max_score = np.max(scores)
    if max_score == 0:
        return scores # Prevent dividing by zero if there are no matches!
    return scores / max_score


# ==========================================
# STEP 4: THE HYBRID BOSS (The Blender)
# ==========================================
def hybrid_search(query, alpha=0.5):
    print(f"\n--- Searching for: '{query}' (Alpha: {alpha}) ---")
    
    # 1. Get raw scores from Detective Dense
    q_dense = dense_machine.transform([query])
    raw_dense_scores = cosine_similarity(q_dense, dense_db)[0]
    
    # 2. Get raw scores from Detective Sparse
    q_sparse = sparse_machine.transform([query])
    raw_sparse_scores = cosine_similarity(q_sparse, sparse_db)[0]
    
    # 3. Normalize both so they are fair (0.0 to 1.0)
    dense_clean = norm(raw_dense_scores)
    sparse_clean = norm(raw_sparse_scores)
    
    # 4. THE MAGIC BLEND FORMULA
    # alpha controls Dense, (1 - alpha) controls Sparse
    final_scores = (alpha * dense_clean) + ((1 - alpha) * sparse_clean)
    
    # 5. Sort the results from highest to lowest
    winning_ids = np.argsort(-final_scores)
    
    # Print the leaderboard
    for rank, doc_id in enumerate(winning_ids):
        score = final_scores[doc_id]
        print(f"Rank {rank+1} | Score: {score:.2f} | Doc: {documents[doc_id]}")


# ==========================================
# STEP 5: RUNNING THE SIMULATION
# ==========================================
user_query = "fast vehicle"

# Scenario A: 50/50 Blend (A balanced approach)
hybrid_search(user_query, alpha=0.5)

# Scenario B: 100% Dense (Only listen to the Meaning Detective)
hybrid_search(user_query, alpha=1.0)

# Scenario C: 100% Sparse (Only listen to the Keyword Detective)
hybrid_search(user_query, alpha=0.0)


--- Searching for: 'fast vehicle' (Alpha: 0.5) ---
Rank 1 | Score: 1.00 | Doc: A fast food delivery vehicle
Rank 2 | Score: 0.45 | Doc: The fastest racing vehicle
Rank 3 | Score: 0.00 | Doc: The quick sports car

--- Searching for: 'fast vehicle' (Alpha: 1.0) ---
Rank 1 | Score: 1.00 | Doc: A fast food delivery vehicle
Rank 2 | Score: 0.39 | Doc: The fastest racing vehicle
Rank 3 | Score: 0.00 | Doc: The quick sports car

--- Searching for: 'fast vehicle' (Alpha: 0.0) ---
Rank 1 | Score: 1.00 | Doc: A fast food delivery vehicle
Rank 2 | Score: 0.50 | Doc: The fastest racing vehicle
Rank 3 | Score: 0.00 | Doc: The quick sports car
